# 01 — Dataset Exploration

**AI-Based Retinal Imaging and Ophthalmic Screening System**

This notebook:
1. Loads the ODIR-5K dataset from HuggingFace
2. Inspects the real dataset structure (columns, dtypes, shapes)
3. Visualises sample images
4. Analyses label distribution
5. Documents the four-class labelling decision

> ⚠️ Run all cells top-to-bottom. First run may take a few minutes to download the dataset.

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
# If running locally in the project venv, skip this cell.
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q datasets huggingface_hub tensorflow keras opencv-python scikit-learn matplotlib seaborn plotly
    # Mount drive if you want to save outputs
    # from google.colab import drive; drive.mount('/content/drive')

print('Dependencies ready.')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Add project root to path (adjust if running from a different location)
ROOT = Path('..').resolve() if not IN_COLAB else Path('/content/retinal_screening_ai')
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

print('Imports OK')

In [ ]:
# ── 1. Load HuggingFace Dataset ────────────────────────────────────────────────
from datasets import load_dataset

print('Loading ODIR-5K dataset from HuggingFace ...')
print('(First run downloads ~600 MB — subsequent runs use cache)')

ds = load_dataset('bumbledeep/odir', split='train')
print(f'\nDataset loaded. Type: {type(ds)}')
print(f'Number of records: {len(ds):,}')

In [ ]:
# ── 2. Inspect Schema ─────────────────────────────────────────────────────────
print('=== Column Schema ===')
for col, feat in ds.features.items():
    print(f'  {col:<20} {feat}')

In [ ]:
# ── 3. Sample Rows ────────────────────────────────────────────────────────────
print('=== First 3 Records ===')
for i in range(3):
    row = ds[i]
    print(f'\nRow {i}:')
    for k, v in row.items():
        if k == 'image':
            print(f'  {k:<15} PIL.Image size={v.size} mode={v.mode}')
        else:
            print(f'  {k:<15} {repr(v)}')

In [ ]:
# ── 4. Build Metadata DataFrame ───────────────────────────────────────────────
df = ds.to_pandas().drop(columns=['image'])
print(f'DataFrame shape: {df.shape}')
print(f'\nColumn dtypes:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())
df.head(10)

In [ ]:
# ── 5. Unique Labels ──────────────────────────────────────────────────────────
print(f'Total unique label strings: {df["label"].nunique()}')
print(f'\nTop 30 most common labels:')
print(df['label'].value_counts().head(30).to_string())

In [ ]:
# ── 6. Four-Class Label Assignment ────────────────────────────────────────────
KEYWORD_MAP = {
    'Normal':   ['normal'],
    'Diabetes': ['diabetes', 'diabetic retinopathy', 'dr'],
    'Glaucoma': ['glaucoma'],
    'AMD':      ['age-related macular degeneration', 'amd', 'macular degeneration'],
}

TARGET_CLASSES = ['Normal', 'Diabetes', 'Glaucoma', 'AMD']

def match_label(label_text):
    if not isinstance(label_text, str):
        return []
    lower = label_text.lower()
    return [cls for cls, kws in KEYWORD_MAP.items() if any(kw in lower for kw in kws)]

df['matched_classes'] = df['label'].apply(match_label)

no_match_count  = (df['matched_classes'].apply(len) == 0).sum()
single_count    = (df['matched_classes'].apply(len) == 1).sum()
multi_count     = (df['matched_classes'].apply(len) > 1).sum()

print(f'Total records : {len(df):,}')
print(f'No match      : {no_match_count:,}  → excluded')
print(f'Single class  : {single_count:,}  → kept')
print(f'Multi-class   : {multi_count:,}   → excluded (policy: exclude)')

In [ ]:
# ── 7. Filter to four classes ──────────────────────────────────────────────────
df['class_name'] = df['matched_classes'].apply(
    lambda m: m[0] if len(m) == 1 else None
)
df_clean = df[df['class_name'].isin(TARGET_CLASSES)].copy()
df_clean['class_index'] = df_clean['class_name'].apply(TARGET_CLASSES.index)

print(f'Records after filtering: {len(df_clean):,}')
print(f'\nClass distribution:')
print(df_clean['class_name'].value_counts())

In [ ]:
# ── 8. Class Distribution Plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df_clean['class_name'].value_counts().reindex(TARGET_CLASSES, fill_value=0)
palette = ['#4CAF50', '#F44336', '#2196F3', '#FF9800']

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=palette, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_title('Four-Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].spines[['top', 'right']].set_visible(False)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=palette,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.suptitle('ODIR-5K — Four-Class Dataset Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ── 9. Sample Images per Class ────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, wspace=0.1, hspace=0.3)

class_colors = {'Normal': '#4CAF50', 'Diabetes': '#F44336',
                'Glaucoma': '#2196F3', 'AMD': '#FF9800'}

for col_idx, cls in enumerate(TARGET_CLASSES):
    # Find HF indices for this class
    idxs = df_clean[df_clean['class_name'] == cls].index.tolist()
    for row_idx in range(2):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        if row_idx < len(idxs):
            hf_idx = idxs[row_idx]
            img = ds[int(hf_idx)]['image'].convert('RGB').resize((224, 224))
            ax.imshow(img)
            if row_idx == 0:
                ax.set_title(cls, fontsize=13, fontweight='bold',
                             color=class_colors[cls], pad=8)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(class_colors.get(cls, 'gray'))
            spine.set_linewidth(2)
            spine.set_visible(True)

plt.suptitle('Sample Retinal Images — One Per Class (ODIR-5K)',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('../reports/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample images saved.')

In [ ]:
# ── 10. Age & Sex Distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age histogram per class
for cls, color in class_colors.items():
    subset = df_clean[df_clean['class_name'] == cls]['age'].dropna()
    axes[0].hist(subset, bins=20, alpha=0.5, label=cls, color=color)
axes[0].set_title('Age Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

# Sex distribution
sex_counts = df_clean['sex'].value_counts()
axes[1].pie(sex_counts.values, labels=sex_counts.index,
            autopct='%1.1f%%', colors=['#64B5F6', '#EF9A9A'],
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Sex Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Demographic Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('Done.')

## Summary

| Item | Value |
|---|---|
| Total records | *(printed above)* |
| Records in four-class prototype | *(printed above)* |
| Multi-label exclusions | *(printed above)* |
| Classes | Normal, Diabetes, Glaucoma, AMD |
| Split policy | Patient-level (no leakage) |

**Next notebook:** `02_preprocessing.ipynb`